<a href="https://colab.research.google.com/github/BarisEmreRuzgar/adli-fotorobot-claude/blob/main/Adli_Fotorobot_V3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔍 Adli Fotorobot v4

Tanık ifadesinden fotogerçekçi fotorobot üretimi ve kimlik korumalı düzenleme.

**Başlamadan önce:**
1. `Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU` seçin (ücretsiz T4 yeterlidir, A100 gerekmez).
2. *(İsteğe bağlı)* Soldaki 🔑 **Secrets** panelinden `GOOGLE_API_KEY` adıyla [Gemini API anahtarınızı](https://aistudio.google.com/apikey) ekleyin ve not defterine erişim izni verin. Anahtar yoksa sistem kural tabanlı Türkçe analizle çalışır.
3. Hücreleri sırayla çalıştırın. Son hücre bir `https://....gradio.live` bağlantısı verir.

In [ ]:
# 1) GPU kontrolü + proje kodu + kütüphaneler (~3-4 dk)
import os, subprocess, sys

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
print("GPU:", gpu.stdout.strip() or "YOK! Çalışma zamanı türünü T4 GPU yapın.")

REPO = "https://github.com/BarisEmreRuzgar/adli-fotorobot-claude.git"
if os.path.isdir("/content/adli-fotorobot-claude/.git"):
    subprocess.run(["git", "-C", "/content/adli-fotorobot-claude", "pull", "-q"], check=False)
else:
    subprocess.run(["git", "clone", "-q", REPO, "/content/adli-fotorobot-claude"], check=True)
os.chdir("/content/adli-fotorobot-claude")

# Colab'ın hazır PyTorch'u kullanılır; insightface'in CPU onnxruntime bağımlılığı
# GPU sürümüyle çakışmasın diye --no-deps ile kurulur.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "onnxruntime"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "insightface==1.0.1"], check=True)
print("✅ Kurulum tamam")

In [ ]:
# 2) Gemini anahtarı (Colab Secrets → GOOGLE_API_KEY). Yoksa kural tabanlı analiz kullanılır.
import os
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY") or ""
except Exception:
    pass
print("🤖 Gemini:", "anahtar bulundu" if os.environ.get("GOOGLE_API_KEY") else "anahtar yok → kural tabanlı analiz")

In [ ]:
# 3) Uygulamayı başlat. İlk çalıştırmada modeller indirilir (~7 GB, birkaç dakika).
#    Aşağıda çıkan https://....gradio.live bağlantısına tıklayın. Durdurmak için hücreyi durdurun.
%cd /content/adli-fotorobot-claude
!python app.py --share

### 💡 İpuçları
- **Aşama 1:** Tanık ifadesini yazın → *Analiz et ve üret*. Çıkan özellik tablosunu düzeltip *Tablodan üret* ile yeniden üretebilirsiniz.
- **Aşama 2:** Bir yüz seçip gönderin, her satıra bir değişiklik yazın (`sakalı temizle`, `gözlük ekle`, `10 yaş yaşlandır`).
  - *Kimlik koruma gücü* yüksekse yüz daha az değişir; *Değişim miktarı* yüksekse poz ve ışık da yeniden üretilir.
  - Her sonuçta **kimlik benzerliği** yazar (ArcFace, %50 üzeri genelde aynı kişi).
- **Vaka Kaydı:** Tüm görseller, prompt'lar ve tohumlar `ciktilar/` klasörüne kaydedilir; ZIP olarak indirilebilir.